# Week 8 Class Project: The Smart Vitals API
**CMS22301 Intelligent Systems and Emerging Technologies · Ravensbourne University London · about 2 hours, in class**

Your Smart Vitals model works in a notebook. Today you put it **behind an API**, so the simulator, the alert service and the clinician dashboard can all use it, and you test what happens when an hour of real-world readings arrives: dropouts, impossible values, late readings and resent copies.

By the end you will have:
- an API that **validates**, **classifies**, **stores** and **alerts** (requirements 2, 4, 6 and 9 of the brief)
- four small functions you wrote yourself, every check saying PASS
- measured evidence: duplicate alerts, false alerts and response times for four designs
- an API contract table and **ADR 2** for your portfolio

**How to run:** Jupyter (`Shift + Enter` runs a cell) or Google Colab (File → Upload notebook). Run the cells **in order from the top**. Flask, scikit-learn and pandas are already installed in Colab and Anaconda.
Lines marked `# TODO` contain `___` blanks for you to complete. The CHECK cells print **PASS**, **TODO** (not finished yet) or **FIX** (with a hint).
If something breaks: *Kernel → Restart & Run All* (Jupyter) or *Runtime → Run all* (Colab).

All data is **simulated for teaching**; no real patient data is used.

## Setup
Run this cell first (about 10 seconds). It trains the model, creates 20 patients and their clinicians, simulates the hour of readings, and defines the API. The API calls **your** functions, so most of it will say TODO until Step 2 is done.

In [ ]:
# SETUP: run this cell first. You do not need to change it.
# It trains a Smart Vitals model, creates 20 patients, simulates one hour of readings
# (with the faults real sensors produce) and defines the API, which uses YOUR four functions.
import json, sqlite3, time, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from flask import Flask, request, jsonify
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

FEATURES = ["risk_level", "diabetic", "heart_rate", "spo2", "systolic_bp", "diastolic_bp", "temperature", "glucose"]
VITALS = FEATURES[2:]
LIMITS = {"heart_rate": (30, 220), "spo2": (70, 100), "systolic_bp": (60, 250),          # possible values (Week 4):
          "diastolic_bp": (30, 150), "temperature": (30, 43), "glucose": (20, 600)}       # outside = sensor error
MODEL_PATH = "smart_vitals_model.joblib"


def teaching_status(heart, spo2, systolic, temp, glucose):
    """The points rule used to label the practice data (invented for teaching; not a clinical score)."""
    points = (np.where((heart < 50) | (heart > 120), 2, np.where(heart > 100, 1, 0))
              + np.where(spo2 < 91, 3, np.where(spo2 < 94, 1, 0))
              + np.where((systolic < 90) | (systolic > 200), 3, np.where((systolic < 100) | (systolic > 170), 1, 0))
              + np.where((temp > 39) | (temp < 35.5), 2, np.where(temp > 38, 1, 0))
              + np.where(np.isnan(glucose), 0, np.where((glucose < 70) | (glucose > 300), 3, np.where(glucose > 220, 1, 0))))
    return np.where(points >= 3, "Critical", np.where(points >= 1, "Warning", "Normal"))


def vitals(rs, n, base_hr, base_sys, diabetic, k):
    """n readings for one patient; k = how unwell (0 = well)."""
    heart = base_hr + rs.normal(0, 6, n) + k * rs.choice([-22, 38], n)
    spo2 = np.minimum(100, rs.normal(97, 1.1, n) - k * rs.uniform(2, 7, n))
    systolic = base_sys + rs.normal(0, 8, n) + k * rs.choice([-35, 55], n)
    diastolic = 0.55 * systolic + rs.normal(12, 5, n)
    temp = rs.normal(36.8, 0.3, n) + k * rs.uniform(0, 2.2, n)
    glucose = rs.normal(135, 25, n) + k * rs.choice([-60, 150], n) if diabetic else np.full(n, np.nan)
    return heart, spo2, systolic, diastolic, temp, glucose


def train_model(patients=60, readings=30, seed=42):
    """The Smart Vitals model from the project starter, trained on 1,800 simulated readings."""
    rs = np.random.RandomState(seed)
    rows = []
    for p in range(patients):
        risk, diabetic = rs.choice([1, 2, 3], p=[.5, .3, .2]), int(rs.rand() < 0.35)
        base_hr, base_sys = rs.normal(72, 7), rs.normal(118, 9) + 6 * risk
        unwell = rs.rand(readings) < 0.10 * risk + 0.06
        k = np.where(unwell, rs.uniform(0.15, 1.5, readings), 0.0)
        hr, sp, sy, di, te, gl = vitals(rs, readings, base_hr, base_sys, diabetic, k)
        lab = teaching_status(hr, sp, sy, te, gl)
        rows.append(pd.DataFrame({"risk_level": risk, "diabetic": diabetic, "heart_rate": hr, "spo2": sp, "systolic_bp": sy,
                                  "diastolic_bp": di, "temperature": te, "glucose": gl, "status": lab}))
    data = pd.concat(rows, ignore_index=True)
    model = make_pipeline(SimpleImputer(strategy="median"),
                          RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42))
    return model.fit(data[FEATURES], data["status"])


def make_patients(n=20, seed=8):
    """20 registered patients, each with a risk level, diabetes status and an assigned clinician."""
    rs = np.random.RandomState(seed)
    return {f"P{i:03d}": {"risk_level": int(rs.choice([1, 2, 3], p=[.5, .3, .2])), "diabetic": int(rs.rand() < 0.35),
                          "clinician": f"C0{i % 4 + 1}", "base_hr": float(rs.normal(72, 7)), "base_sys": float(rs.normal(118, 9))}
            for i in range(1, n + 1)}


def make_stream(minutes=60, seed=8, dup_rate=0.03, glitch_rate=0.01, dropout_rate=0.03, delay_rate=0.02, strangers=3):
    """One hour of readings from every patient, in the order they ARRIVE at the cloud, with real-world faults:
    dropouts (a vital missing), glitches (an impossible value), late readings, resent copies and unknown patients."""
    rs = np.random.RandomState(seed)
    start = datetime(2026, 11, 10, 9, 0)
    stream = []
    for pid, p in PATIENTS.items():
        k = np.zeros(minutes)
        if rs.rand() < 0.10 * p["risk_level"] + 0.05:                       # some patients have an unwell spell
            s0 = rs.randint(5, minutes - 20)
            k[s0:s0 + rs.randint(8, 20)] = rs.uniform(0.6, 1.4)
        k = np.where((k == 0) & (rs.rand(minutes) < 0.06), rs.uniform(0.15, 0.8, minutes), k)
        hr, sp, sy, di, te, gl = vitals(rs, minutes, p["base_hr"], p["base_sys"] + 6 * p["risk_level"], p["diabetic"], k)
        truth = teaching_status(hr, sp, sy, te, gl)
        for m in range(minutes):
            t = start + timedelta(minutes=m)
            r = {"reading_id": f"{pid}-{m + 1:03d}", "patient_id": pid, "time": t.isoformat(timespec="seconds"),
                 "heart_rate": round(float(hr[m])), "spo2": round(float(sp[m]), 1), "systolic_bp": round(float(sy[m])),
                 "diastolic_bp": round(float(di[m])), "temperature": round(float(te[m]), 1),
                 "glucose": None if np.isnan(gl[m]) else round(float(gl[m])),
                 "_truth": str(truth[m]), "_fault": "", "_arrives": t + timedelta(seconds=float(rs.uniform(0.2, 2)))}
            u = rs.rand()
            if u < glitch_rate:                                               # an impossible value: a sensor error
                name, bad = [("heart_rate", 250), ("spo2", 130.0), ("temperature", 50.0)][rs.randint(3)]
                r[name], r["_truth"], r["_fault"] = bad, "sensor error", "glitch"
            elif u < glitch_rate + dropout_rate:                              # one vital dropped out
                r[["heart_rate", "spo2", "systolic_bp", "temperature"][rs.randint(4)]] = None
                r["_fault"] = "dropout"
            if rs.rand() < delay_rate:                                        # arrives five minutes late
                r["_arrives"] += timedelta(minutes=5); r["_fault"] = (r["_fault"] + " late").strip()
            stream.append(r)
            if rs.rand() < dup_rate:                                          # the phone resends it (MQTT QoS 1)
                copy = dict(r); copy["_arrives"] = r["_arrives"] + timedelta(seconds=float(rs.uniform(5, 15)))
                copy["_fault"] = (r["_fault"] + " resend").strip()
                stream.append(copy)
    for i in range(strangers):                                                # readings for an unregistered patient
        t = start + timedelta(minutes=int(rs.randint(minutes)))
        stream.append({"reading_id": f"P099-{i + 1:03d}", "patient_id": "P099", "time": t.isoformat(timespec="seconds"),
                       "heart_rate": 80, "spo2": 97.0, "systolic_bp": 120, "diastolic_bp": 78, "temperature": 36.8, "glucose": None,
                       "_truth": "unknown patient", "_fault": "stranger", "_arrives": t + timedelta(seconds=1)})
    return sorted(stream, key=lambda r: r["_arrives"])


def create_app(model, validate=True, dedupe=True, check_access=False, load_each_time=False):
    """The Smart Vitals API. It calls your functions: problems_in, already_stored, make_alert, status_code (and can_view)."""
    app = Flask("smart_vitals")
    app.config["TESTING"] = True
    db = sqlite3.connect(":memory:", check_same_thread=False)     # the API keeps no state itself: it lives in the database
    db.execute("CREATE TABLE readings (reading_id TEXT, patient_id TEXT, time TEXT, status TEXT, data TEXT)")
    db.execute("CREATE TABLE alerts (alert_id INTEGER PRIMARY KEY, reading_id TEXT, patient_id TEXT, clinician_id TEXT, data TEXT)")
    app.db = db

    @app.post("/readings")
    def receive_reading():
        reading = request.get_json()
        if validate:
            problems = problems_in(reading)
            if problems:
                return jsonify({"outcome": "invalid", "problems": problems}), status_code("invalid")
        patient = PATIENTS.get(reading.get("patient_id"))
        if patient is None:
            return jsonify({"outcome": "unknown patient"}), status_code("unknown patient")
        if dedupe and already_stored(db, reading["reading_id"]):
            return jsonify({"outcome": "duplicate", "reading_id": reading["reading_id"]}), status_code("duplicate")
        m = joblib.load(MODEL_PATH) if load_each_time else model
        row = pd.DataFrame([{"risk_level": patient["risk_level"], "diabetic": patient["diabetic"],
                             **{v: reading.get(v) for v in VITALS}}], columns=FEATURES).astype(float)
        status = str(m.predict(row)[0])
        db.execute("INSERT INTO readings VALUES (?, ?, ?, ?, ?)",
                   (reading["reading_id"], reading["patient_id"], reading["time"], status, json.dumps(reading)))
        alert = None
        if status == "Critical":
            alert = make_alert(reading, status, patient)
            db.execute("INSERT INTO alerts (reading_id, patient_id, clinician_id, data) VALUES (?, ?, ?, ?)",
                       (reading["reading_id"], reading["patient_id"], alert["clinician_id"], json.dumps(alert)))
        return jsonify({"outcome": "created", "reading_id": reading["reading_id"], "status": status,
                        "alert": alert is not None}), status_code("created")

    @app.get("/patients/<patient_id>/readings")
    def patient_history(patient_id):
        if check_access and not can_view(request.headers.get("X-Clinician-ID"), PATIENTS.get(patient_id)):
            return jsonify({"outcome": "forbidden"}), 403
        rows = db.execute("SELECT time, status, data FROM readings WHERE patient_id = ? ORDER BY time", (patient_id,)).fetchall()
        return jsonify([{"time": t, "status": s, **{k: v for k, v in json.loads(d).items() if k in VITALS}} for t, s, d in rows])

    @app.get("/alerts")
    def list_alerts():
        rows = db.execute("SELECT data FROM alerts WHERE clinician_id = ?", (request.args.get("clinician", ""),)).fetchall()
        return jsonify([json.loads(d) for (d,) in rows])

    @app.get("/health")
    def health():
        return jsonify({"status": "ok"})

    return app


RESULT_ROWS = ["Requests sent", "Stored (201)", "Duplicates ignored (200)", "Rejected as impossible (422)",
               "Unknown patient (404)", "Alerts sent to clinicians", "Duplicate alerts", "False alerts from sensor glitches",
               "Critical readings with no alert", "Response time: median (ms)", "Response time: p95 (ms)"]


def play(app, stream):
    """Send every reading to the API, in arrival order, and count what happened."""
    client = app.test_client()
    codes, ms = [], []
    for r in stream:
        body = {k: v for k, v in r.items() if not k.startswith("_")}
        t0 = time.perf_counter()
        codes.append(client.post("/readings", json=body).status_code)
        ms.append((time.perf_counter() - t0) * 1000)
    alerted = [rid for (rid,) in app.db.execute("SELECT reading_id FROM alerts")]
    glitches = {r["reading_id"] for r in stream if "glitch" in r["_fault"]}
    critical = {r["reading_id"] for r in stream if r["_truth"] == "Critical"}
    return {"_missed": sorted(critical - set(alerted)), "_codes": codes, "_ms": ms,"Requests sent": len(stream), "Stored (201)": codes.count(201), "Duplicates ignored (200)": codes.count(200),
            "Rejected as impossible (422)": codes.count(422), "Unknown patient (404)": codes.count(404),
            "Alerts sent to clinicians": len(alerted), "Duplicate alerts": len(alerted) - len(set(alerted)),
            "False alerts from sensor glitches": sum(rid in glitches for rid in alerted),
            "Critical readings with no alert": len(critical - set(alerted)),
            "Response time: median (ms)": round(float(np.median(ms)), 1), "Response time: p95 (ms)": round(float(np.percentile(ms, 95)), 1)}


def compare(designs):
    """designs: {name: results}. Print one table."""
    if any(isinstance(v, str) for v in designs.values()):      # "TODO" or "FIX": that design could not run
        print("Some designs could not run: finish or fix your functions in Step 2, then run this cell again")
        designs = {k: v for k, v in designs.items() if not isinstance(v, str)}
    if designs:
        show = lambda x: f"{x:,}" if isinstance(x, int) else f"{x:.1f}"
        table = pd.DataFrame({k: [show(v[r]) for r in RESULT_ROWS] for k, v in designs.items()}, index=RESULT_ROWS)
        print(table.to_string())


STEPS = {}


def check(name, test):
    """Run one test and say PASS, TODO or FIX."""
    try:
        test()
        print(f"PASS  {name}")
    except NameError as problem:
        print(f"TODO  {name}: replace every ___ with your code" if "'___'" in str(problem)
              else f"FIX   {name}: {problem} (run the cells above in order)")
    except AssertionError as problem:
        print(f"FIX   {name}: {problem}")
    except Exception as problem:
        print(f"FIX   {name}: Python stopped with {type(problem).__name__}: {str(problem)[:200]}")


def attempt(step, action):
    """Run one step; say TODO (a ___ is left) or FIX (a mistake) instead of crashing."""
    try:
        result = action()
        STEPS[step] = True
        return result
    except NameError as problem:
        if "'___'" not in str(problem):
            raise
        STEPS[step] = False
        print(f"TODO  {step}: replace every ___ in your functions, run them, then run this cell again")
        return "TODO"
    except Exception as problem:                      # a mistake inside one of YOUR functions
        STEPS[step] = False
        print(f"FIX   {step}: Python stopped with {type(problem).__name__}: {problem}")
        print("      Look again at your Step 2 functions (the checks under them say which one), fix it, run it, then run this cell again")
        return "FIX"


warnings.filterwarnings("ignore", message="X does not have valid feature names")
model = train_model()
joblib.dump(model, MODEL_PATH)
PATIENTS = make_patients()
stream = make_stream()
print("Model trained and saved:", MODEL_PATH)
print("Patients:", len(PATIENTS), "| clinicians:", sorted({p["clinician"] for p in PATIENTS.values()}))
print("Readings arriving in the hour:", len(stream))

## Step 0 · Predict
Before you run anything, write your predictions in the table on your sheet: how many **false alerts** will a sensor glitch cause? How many **duplicate alerts** will resent copies cause? Will checking every reading make the API **slower**?

## Step 1 · Look at the hour
What will the API receive? Notice the faults, and the reading that arrives late.

In [ ]:
import collections
faults = collections.Counter(f for r in stream for f in r["_fault"].split())
print("Readings arriving in the hour:", len(stream), "from", len(PATIENTS), "registered patients (plus some from an unknown one)")
print("Faults built in:", dict(faults))
print("What each reading really is:", dict(collections.Counter(r["_truth"] for r in stream)))
print("\nThe first reading, exactly as the phone sends it:")
print({k: v for k, v in stream[0].items() if not k.startswith("_")})
late = next(r for r in stream if "late" in r["_fault"])
print("\nA late reading: measured at", late["time"][11:], "but arrived at", late["_arrives"].strftime("%H:%M:%S"))

**Q1.** How many readings carry a fault of some kind? Which fault worries you most for a clinician, and why?

**Your answer:**

_(double-click to type)_

## Step 2 · Build four functions
Each function follows a worked example. Replace every `___`, run the cell, then run its CHECK cell. **Every check must say PASS** before Step 3: the API uses your functions.

### 2a · Reject impossible readings (validation)

In [ ]:
# Worked example: a room-temperature sensor (Week 4) can only read -40 to 85 degrees C
ROOM_LIMITS = {"celsius": (-40, 85)}

def room_problems(reading):
    problems = []
    for name, (low, high) in ROOM_LIMITS.items():
        value = reading.get(name)
        if value is None:                      # nothing to check
            continue
        if not (low <= value <= high):         # outside what the sensor can measure
            problems.append(f"{name} = {value} is outside {low} to {high}")
    if not reading.get("room"):
        problems.append("room is missing")
    return problems

print(room_problems({"room": "G01", "celsius": 21.5}))
print(room_problems({"room": "G01", "celsius": 150}))
print(room_problems({"celsius": None}))

In [ ]:
def problems_in(reading):
    """A list of reasons to reject a reading (an empty list means it is fine)."""
    problems = []
    for name, (low, high) in LIMITS.items():
        value = reading.get(name)
        if value is None:            # a missing vital is allowed: the model fills the gap (requirement 9)
            continue
        if not (___ <= value <= ___):                                 # TODO: the two limits, as in the worked example
            problems.append(f"{name} = {value} is outside {low} to {high}")
    if not reading.get(___):                                          # TODO: which field says whose reading this is?
        problems.append("patient_id is missing")
    return problems

In [ ]:
# CHECK
good = {k: v for k, v in stream[0].items() if not k.startswith("_")}

def test_good():
    assert problems_in(good) == [], f"a normal reading should have no problems, got {problems_in(good)}"

def test_glitch():
    p = problems_in({**good, "heart_rate": 250})
    assert len(p) == 1 and "heart_rate" in p[0], f"a heart rate of 250 should give one problem, got {p}"

def test_missing_vital():
    assert problems_in({**good, "spo2": None}) == [], "a missing vital is allowed (requirement 9)"

def test_no_patient():
    assert problems_in({**good, "patient_id": None}) == ["patient_id is missing"], "a reading with no patient must be rejected"

check("normal reading accepted", test_good)
check("impossible value rejected", test_glitch)
check("missing vital allowed", test_missing_vital)
check("reading with no patient rejected", test_no_patient)

### 2b · Recognise a resent copy (idempotency)
In Week 7 a Python set remembered message IDs. An API may run as several copies, so the memory must live in the **shared database** instead.

In [ ]:
# Worked example (Week 7): remember message IDs in a Python set
seen = set()
def is_duplicate(msg_id, seen):
    if msg_id in seen:
        return True
    seen.add(msg_id)
    return False
print("Week 7 set:", is_duplicate("A1", seen), is_duplicate("A1", seen))

# The same question asked of a database table: is there already a row with this ID?
demo_db = sqlite3.connect(":memory:")
demo_db.execute("CREATE TABLE messages (msg_id TEXT)")
demo_db.execute("INSERT INTO messages VALUES ('A1')")
print("Row for A1:", demo_db.execute("SELECT 1 FROM messages WHERE msg_id = ?", ("A1",)).fetchone())
print("Row for B7:", demo_db.execute("SELECT 1 FROM messages WHERE msg_id = ?", ("B7",)).fetchone())

In [ ]:
def already_stored(db, reading_id):
    """True if this reading was stored before: a resent copy."""
    row = db.execute("SELECT 1 FROM readings WHERE reading_id = ?", (___,)).fetchone()   # TODO: which ID are we looking for?
    return row is not ___                                                                  # TODO: what does fetchone() give when there is no row?

In [ ]:
# CHECK
def test_stored():
    t = sqlite3.connect(":memory:")
    t.execute("CREATE TABLE readings (reading_id TEXT, patient_id TEXT, time TEXT, status TEXT, data TEXT)")
    t.execute("INSERT INTO readings VALUES ('P001-001', 'P001', '', '', '')")
    assert already_stored(t, "P001-001") is True, "a stored reading should give True"
    assert already_stored(t, "P001-002") is False, "a new reading should give False"

check("already_stored finds copies", test_stored)

### 2c · Build the alert, and choose the status codes

In [ ]:
# Worked example: build a message (a dictionary) from a reading
def make_receipt(reading):
    return {"patient_id": reading["patient_id"], "time": reading["time"], "received": True}

print(make_receipt(stream[0]))

# HTTP status codes are numbers with agreed meanings (RFC 9110)
for code, meaning in {200: "OK", 201: "Created", 403: "Forbidden", 404: "Not Found",
                      422: "Unprocessable Content", 500: "Internal Server Error"}.items():
    print(code, meaning)

In [ ]:
def make_alert(reading, status, patient):
    """The alert the brief asks for (requirement 4): which patient, which clinician, the vitals and the time."""
    return {"patient_id": reading["patient_id"],
            "clinician_id": ___,                   # TODO: the patient's assigned clinician (look inside patient)
            "time": ___,                           # TODO: when the reading was taken
            "status": status,
            "vitals": {v: reading.get(v) for v in VITALS}}


def status_code(outcome):
    """The HTTP status code the API sends back for each outcome."""
    codes = {"created": ___,                       # TODO: a new reading was stored
             "duplicate": ___,                     # TODO: a copy we already have: fine, nothing new created
             "invalid": ___,                       # TODO: the reading breaks the rules (Unprocessable Content)
             "unknown patient": ___}               # TODO: no such patient (Not Found)
    return codes[outcome]

In [ ]:
# CHECK
def test_alert():
    r = {k: v for k, v in stream[0].items() if not k.startswith("_")}
    a = make_alert(r, "Critical", PATIENTS[r["patient_id"]])
    assert a["clinician_id"] == PATIENTS[r["patient_id"]]["clinician"], "clinician_id should come from patient['clinician']"
    assert a["time"] == r["time"], "time should be the time the reading was taken"

def test_codes():
    got = {o: status_code(o) for o in ["created", "duplicate", "invalid", "unknown patient"]}
    assert got == {"created": 201, "duplicate": 200, "invalid": 422, "unknown patient": 404}, f"check the codes: {got}"

check("alert has patient, clinician, vitals and time", test_alert)
check("status codes", test_codes)

## Step 3 · Talk to your API
The test client sends real HTTP requests to your app, without starting a server. Read each reply: the **status code** and the **JSON body**.

In [ ]:
def talk_to_the_api():
    app = create_app(model)
    client = app.test_client()                   # sends real HTTP requests to the app, without a server
    one = {k: v for k, v in stream[0].items() if not k.startswith("_")}
    tries = [("A normal reading", one), ("The same reading again", one),
             ("A heart rate of 250", {**one, "reading_id": "TEST-1", "heart_rate": 250}),
             ("A missing SpO2", {**one, "reading_id": "TEST-2", "spo2": None}),
             ("An unknown patient", {**one, "reading_id": "TEST-3", "patient_id": "P099"})]
    for label, body in tries:
        reply = client.post("/readings", json=body)
        print(f"POST {label:<24} -> {reply.status_code} {reply.get_json()}")
    history = client.get(f"/patients/{one['patient_id']}/readings").get_json()
    print(f"GET  history of {one['patient_id']:<15} -> {len(history)} readings, first: {history[0] if history else 'none stored'}")
    print("GET  /health                    ->", client.get("/health").get_json())

attempt("talk to the API", talk_to_the_api);

**Q2.** Why does the second copy of the same reading get a 200 and not a 201? Why does the missing SpO2 still get a 201?

**Your answer:**

_(double-click to type)_

## Step 4 · Play the hour through three designs
- **A · no checks:** every reading goes straight to the model.
- **B · validation:** impossible readings are rejected (422).
- **C · validation + idempotency:** resent copies are recognised (200) and stored once.

Each design takes about 10 seconds. Copy the results into the table on your sheet.

In [ ]:
designs = {
    "A: no checks": attempt("design A", lambda: play(create_app(model, validate=False, dedupe=False), stream)),
    "B: validation": attempt("design B", lambda: play(create_app(model, dedupe=False), stream)),
    "C: validation + idempotency": attempt("design C", lambda: play(create_app(model), stream)),
}
compare(designs)

**Q3.** Compare with your predictions. Where do A's extra alerts come from? Does validation make the API noticeably slower?

**Your answer:**

_(double-click to type)_

## Step 5 · Break it
### 5a · A poor signal: 1 reading in 5 is resent

In [ ]:
flaky = make_stream(dup_rate=0.20)              # a poor signal: 1 reading in 5 is resent
print("Resent copies in the flaky hour:", sum("resend" in r["_fault"] for r in flaky))
compare({"A: no checks": attempt("flaky A", lambda: play(create_app(model, validate=False, dedupe=False), flaky)),
         "C: validation + idempotency": attempt("flaky C", lambda: play(create_app(model), flaky))})

**Q4.** How many clinicians would be called twice in an hour without idempotency? MedLink plans 10,000 patients, not 20: scale your answer up.

**Your answer:**

_(double-click to type)_

### 5b · Load the model on every request
A common first attempt: load the model inside the request handler. Design D does exactly that (about 30 seconds).

In [ ]:
slow = attempt("design D", lambda: play(create_app(model, load_each_time=True), stream))
compare({"C: model loaded once": designs["C: validation + idempotency"], "D: model loaded every request": slow})
if isinstance(slow, dict) and isinstance(designs["C: validation + idempotency"], dict):
    fig, ax = plt.subplots(figsize=(7, 2.6))
    ax.hist(designs["C: validation + idempotency"]["_ms"], bins=60, range=(0, 60), alpha=0.75, color="#0F9B9B", label="C: loaded once")
    ax.hist(slow["_ms"], bins=60, range=(0, 60), alpha=0.6, color="#D9622B", label="D: loaded every request")
    ax.set_xlabel("response time (ms)"); ax.set_ylabel("requests"); ax.legend(frameon=False)
    plt.show()

**Q5.** How much slower is D at the median and at p95? Why would the difference grow with a bigger model?

**Your answer:**

_(double-click to type)_

### 5c · Who can see what?
Requirement 5: clinicians may see only their own patients. Asking for another patient's records by changing the ID in the address is the **most common API vulnerability** (OWASP API Security Top 10, 2023: *Broken Object Level Authorization*).

In [ ]:
# Who can see what? Clinicians send their ID in a header. The API does not check it yet.
def peek():
    app = create_app(model)
    client = app.test_client()
    for r in stream[:200]:
        client.post("/readings", json={k: v for k, v in r.items() if not k.startswith("_")})
    owner = PATIENTS["P003"]["clinician"]
    other = next(c for c in ["C01", "C02", "C03", "C04"] if c != owner)
    for who in [owner, other]:
        reply = client.get("/patients/P003/readings", headers={"X-Clinician-ID": who})
        print(f"{who} asks for P003 (P003's clinician is {owner}) -> {reply.status_code}, {len(reply.get_json())} readings returned")

attempt("who can see what", peek);

Write the check. Then the API refuses with **403 Forbidden**.

In [ ]:
def can_view(clinician_id, patient):
    """A clinician may see only their own patients (requirement 5)."""
    return patient is not None and patient["clinician"] == ___      # TODO: who is asking?

In [ ]:
# CHECK, then try it with the check switched on
def test_can_view():
    p = PATIENTS["P003"]
    assert can_view(p["clinician"], p) is True, "the patient's own clinician should be allowed"
    assert can_view("C99", p) is False, "any other clinician should be refused"

check("clinicians see only their own patients", test_can_view)

def peek_secure():
    client = create_app(model, check_access=True).test_client()
    owner = PATIENTS["P003"]["clinician"]
    other = next(c for c in ["C01", "C02", "C03", "C04"] if c != owner)
    for who in [owner, other]:
        print(f"{who} asks for P003 -> {client.get('/patients/P003/readings', headers={'X-Clinician-ID': who}).status_code}")

attempt("access check", peek_secure);

## Step 6 · Improve: did validation hide a real emergency?
Design C missed a few Critical readings. Look at **why** each one was missed.

In [ ]:
c = designs["C: validation + idempotency"]
if isinstance(c, dict):
    rejected = {r["reading_id"] for r, code in zip(stream, c["_codes"]) if code == 422}
    for rid in c["_missed"]:
        r = next(x for x in stream if x["reading_id"] == rid)
        vit = {k: r[k] for k in ["heart_rate", "spo2", "systolic_bp", "temperature"]}
        print(rid, "| truly", r["_truth"], "|", vit, "| rejected by validation" if rid in rejected else "| the model said otherwise")

One missed reading was **real**, not a glitch: a heart rate below 30 is rare but possible, and the range from Week 4 threw it away. Widen the heart-rate range and play the hour again.

In [ ]:
LIMITS["heart_rate"] = (20, 240)                # a wider range: still rejects the 250 glitches
better = attempt("design E", lambda: play(create_app(model), stream))
compare({"C: heart rate 30 to 220": designs["C: validation + idempotency"], "E: heart rate 20 to 240": better})

**Q6.** Wider limits fixed this case. But a heart rate of 250 can also be real. Propose a better design than rejecting a strange reading silently. (Hint: who should be told, and how?)

**Your answer:**

_(double-click to type)_

## Step 7 · Defend your design
**1 · The API contract.** Complete this table in your notebook and copy it to `docs/api.md` in your repository.

| Method and path | What it does | Success code | Error codes | Requirement |
|---|---|---|---|---|
| `POST /readings` | | | | R2, R4, R9 |
| `GET /patients/{id}/readings` | | | | R6, R5 |
| `GET /alerts?clinician=` | | | | R4 |
| `GET /health` | | | | R11 |

**2 · ADR 2.** One decision from today, for example *"Out-of-range readings are flagged, not dropped"* or *"Readings are idempotent by reading_id"*. Context · Options · Decision · Evidence (your numbers) · Consequences. Save it as `docs/adr/0002-....md`.

**3 · Reflection.** Three lines: what surprised you, what you would test next, what this means for your Smart Vitals submission.

**Your answer:**

_(double-click to type)_

## Finished early?
- Make design C refuse a reading whose `time` is more than an hour in the future. What status code fits?
- Add `PUT /patients/<id>/thresholds` so a clinician can set a patient's own alert limits (requirement 10). What must it check first?
- Time 1,000 requests to `/health`. What does that tell you about where the API spends its time?